This project focuses on a clustering exercise on the trips made with the public bicycles of London (Santander Cycles) in 2025. The idea is to work with the raw data, clean it and transform it into something useful for the clustering exercise. We want to analyse and cluster the stations using 2 different representations: a **vector representation of the stations** and a **graph representation of the flows between stations**. The workflow is the following:

1. Load all the data (multiple .csv files)
2. Clean raw data
3. Analyze the data and transform it
4. Clustering exercise 1: Station analysis without considering the network
    - Transform data to create station profiles
    - Dimension reduction techniques
    - Clustering of data
    - Anomaly detection
    - Model conclusion
5. Clustering exercise 2: Station analysis considering the network
    - Transform data to create origin-destination features
    - Stochastic block model (SBM)
    - Model conclusion
6. Model comparison and final conclusion

All these steps are divided in 2 notebooks. The first one tackles the first 2 points and the second one tackles the other 4.

*[CLAUDE - EDITED: typos fixed; the French sentence copied from the PDF had broken characters ('reprØsentation', 'ux' instead of 'flux') and was translated]*

In [9]:
import polars as pl
from pathlib import Path

# 1. Load all data
After downloading all files, they will all be loaded using Polars and will be merged into one single DF

In [10]:
files_path = Path("../data/input_raw").glob("*.csv")

for file in Path("../data/input_raw").glob("*.csv"):
    df = pl.read_csv(file)

    print(file)
    print(df.columns)
    print("----------------")

# the files have the same column names but not in the same order


..\data\input_raw\411JourneyDataExtract01Jan2025-14Jan2025.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------
..\data\input_raw\412JourneyDataExtract15Jan2025-31Jan2025.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------
..\data\input_raw\413JourneyDataExtract01Feb2025-14Feb2025.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------
..\data\input_raw\414JourneyDataExtract15Feb2025-28Feb2025.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model'

In [11]:
################
# Read the files
################
files = sorted(Path("../data/input_raw").glob("*.csv"))

# Use first file order
first_df = pl.read_csv(files[0])
columns = first_df.columns

#Read and reorder files
dfs = [pl.read_csv(file).select(columns)
       for file in files]

df = pl.concat(dfs)

print(df.head())
print(df.shape)
print("----------------------")
# All dfs are grouped into one giant df of shape (9068241, 11)

shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Number    ┆ Start     ┆ Start     ┆ Start     ┆ … ┆ Bike      ┆ Bike      ┆ Total     ┆ Total    │
│ ---       ┆ date      ┆ station   ┆ station   ┆   ┆ number    ┆ model     ┆ duration  ┆ duration │
│ i64       ┆ ---       ┆ number    ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ (ms)     │
│           ┆ str       ┆ ---       ┆ str       ┆   ┆ i64       ┆ str       ┆ str       ┆ ---      │
│           ┆           ┆ i64       ┆           ┆   ┆           ┆           ┆           ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 145666816 ┆ 2025-01-1 ┆ 1043      ┆ Museum of ┆ … ┆ 55223     ┆ CLASSIC   ┆ 14m 30s   ┆ 870468   │
│           ┆ 4 23:59   ┆           ┆ London,   ┆   ┆           ┆           ┆           ┆          │
│           ┆           ┆           ┆ Barbican  ┆   ┆           ┆           

# 2. Clean raw data
We will then do some basic data cleaning of the whole dataset to make sure that we keep only the necessary information + some basic analysis on it

In [12]:
#########################
#Filter df to keep only useful columns
#######################


columns_to_drop = ["Number", "Start station", "End station", "Bike number", "Bike model", "Total duration"]

df = df.drop(columns_to_drop)

print(df.head())
print(df.columns)
print(df.shape)

shape: (5, 5)
┌──────────────────┬───────────────┬──────────────────┬──────────────────────┬─────────────────────┐
│ Start date       ┆ Start station ┆ End date         ┆ End station number   ┆ Total duration (ms) │
│ ---              ┆ number        ┆ ---              ┆ ---                  ┆ ---                 │
│ str              ┆ ---           ┆ str              ┆ i64                  ┆ i64                 │
│                  ┆ i64           ┆                  ┆                      ┆                     │
╞══════════════════╪═══════════════╪══════════════════╪══════════════════════╪═════════════════════╡
│ 2025-01-14 23:59 ┆ 1043          ┆ 2025-01-15 00:13 ┆ 200149               ┆ 870468              │
│ 2025-01-14 23:59 ┆ 300015        ┆ 2025-01-15 00:04 ┆ 300229               ┆ 307181              │
│ 2025-01-14 23:59 ┆ 1068          ┆ 2025-01-15 00:10 ┆ 1051                 ┆ 640755              │
│ 2025-01-14 23:59 ┆ 1159          ┆ 2025-01-15 00:10 ┆ 1007                 

In [13]:
###################
#Export csv
###################

#df.write_csv("../data/output/v2/1.full_filtered_df.csv")
df.write_parquet("../data/output/v2/1.full_filtered_df.parquet")

One of the basic things to consider is the existence of missing values. There are 184 missing values in the columns 'End station number' and 'Total duration (ms)'. After `drop_nulls()` the dataframe goes from 9,068,241 to 9,068,057 rows, i.e. exactly 184 rows are removed: **the missing values are on the same rows** (trips without an end station also have no duration: most likely bikes that were never docked). Removing them is a loss of 0.002% of the data.

*[CLAUDE - EDITED: the previous text assumed the nulls could be on different rows (184*2 = 368); the output shows they are on the same 184 rows]*

In [14]:
########################
#Missing start or end station number
########################
missing_values = df.null_count()

print(missing_values)

#184 missing values for columns end station number and total duration. if the missing values for the entries are not the same we have a total of 
# 184*2 = 368 out of 9068241 entries. Removing them would be a loss of less than 1% of the dataframe

df_no_nulls = df.drop_nulls()
print(df_no_nulls.shape)
print("---------------")

shape: (1, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ u32        ┆ u32                  ┆ u32      ┆ u32                ┆ u32                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
│ 0          ┆ 0                    ┆ 0        ┆ 184                ┆ 184                 │
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘
(9068057, 5)
---------------


Another point to consider is the "impossible" trips. This means trips that last for 0 seconds or that last for less than that. After removing the missing values of the DF, we can see that there are no negative or 0 second entries on the DF.

In [15]:
########################
#Total duration = 0, missing or negative
########################

df_duration = df_no_nulls.filter(pl.col("Total duration (ms)") <= 0)
print(df_duration)
print("-------------------")

# After removing the missing entries for the total duration, we see there are no negative or 0 ms entries 


shape: (0, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ str        ┆ i64                  ┆ str      ┆ i64                ┆ i64                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘
-------------------


We can also check round trips. A round trip is defined as a trip whose start station is the same as its end station. These trips carry no origin-destination information (they would be self-loops in the flow graph of notebook 2) and we cannot know where the user actually went, so we identify them and remove them from the data.

*[CLAUDE - EDITED: justification of the removal made more precise]*

In [16]:
########################
#same start and end station number
########################

#are there entries where the start and end stations are the same?
df_same_station = df_no_nulls.filter(pl.col("Start station number") == pl.col("End station number"))
print(df_same_station)

#305702 entries start and end in the same station. this represents 305702/9068057 = 0.033 -> 3.3% of the no_nulls dataset

#Lets filter them out
df_no_same_station = df_no_nulls.filter(pl.col("Start station number") != pl.col("End station number"))
print(df_no_same_station)

#new df shape is 8_762_355

shape: (305_702, 5)
┌──────────────────┬───────────────┬──────────────────┬──────────────────────┬─────────────────────┐
│ Start date       ┆ Start station ┆ End date         ┆ End station number   ┆ Total duration (ms) │
│ ---              ┆ number        ┆ ---              ┆ ---                  ┆ ---                 │
│ str              ┆ ---           ┆ str              ┆ i64                  ┆ i64                 │
│                  ┆ i64           ┆                  ┆                      ┆                     │
╞══════════════════╪═══════════════╪══════════════════╪══════════════════════╪═════════════════════╡
│ 2025-01-14 23:40 ┆ 300050        ┆ 2025-01-15 00:06 ┆ 300050               ┆ 1559817             │
│ 2025-01-14 23:35 ┆ 300059        ┆ 2025-01-15 00:46 ┆ 300059               ┆ 4281047             │
│ 2025-01-14 23:35 ┆ 300059        ┆ 2025-01-15 00:46 ┆ 300059               ┆ 4301835             │
│ 2025-01-14 23:27 ┆ 200166        ┆ 2025-01-14 23:27 ┆ 200166         

We can see that 305702 entries have the same start and end station. This represents 305702/9068057 = 0.033 -> 3.3% of the entries in the dataset (after removing the missing values). We will remove these round trips and we will end up with a dataframe with 8762355 entries in the end.


It is also important to check the consistency of the start and end stations. A good way to check this is making sure that all stations appearing as start station are also appearing as end stations and vice versa. And since we are checking the ids here, we can also check the consistency of these by making sure we have no negative ids.

In [ ]:
########################
#incoherent start and end station numbers (?)
########################

#-------------------------
#Stations appearing in one column but not the other?
#-------------------------

#get uniqueids for both columns
start_ids = df_no_same_station["Start station number"].unique()
end_ids = df_no_same_station["End station number"].unique()

#compare them
# [CLAUDE - CHANGED] .implode() added: is_in(<Series>) is deprecated in recent Polars (it raised a DeprecationWarning)
start_only = start_ids.filter(~start_ids.is_in(end_ids.implode()))

end_only = end_ids.filter(~end_ids.is_in(start_ids.implode()))


print("Only in Start:", start_only)
print("Only in End:", end_only)

#There are 2 stations that only appear in the end station number column, station ids 10626 and 22168. They are removed a few cells below.

#-------------------------
# Negative station numbers?
#-------------------------

negative_stations = df_no_same_station.filter((pl.col("Start station number") < 0) | (pl.col("End station number") < 0))

print(negative_stations)

# No negative station number ids

It is seen that there are no negative station identifiers. On the other hand, there are 2 stations that only appear in the end station number column: station ids 10626 and 22168.

*[CLAUDE - EDITED: typo; the question 'should they be removed??' is answered in the next cell]*

> **[CLAUDE - ADDED]** Answer to the question above: **yes, they should be removed.**
>
> Stations 10626 and 22168 only ever appear as an *arrival* station (607 and 199 trips respectively), they are **not** in `tfl_bike_stations.csv` (so they have no name and no coordinates), and they cannot be described by departure-based features (number of departures, mean duration of departing trips, rush-hour ratios...). If they are kept, the station profile table in notebook 2 gets two rows with `n_departures = 0` and, because of the `full` join, their station id becomes `0` — the "station 0" that kept showing up as an outlier. The most likely explanation is that they are decommissioned / temporary docking points where bikes could still be returned.
>
> We therefore remove **every trip that starts or ends at these stations** (806 trips, < 0.01% of the data). The code is generic: it removes any station found in `start_only` or `end_only`, so it keeps working if the raw files change.

In [ ]:
# [CLAUDE - ADDED] ------------------------------------------------------------
# Remove the stations that only appear in one of the two columns (10626 and 22168)
# -----------------------------------------------------------------------------
stations_to_remove = end_only.to_list() + start_only.to_list()
print("Stations removed:", stations_to_remove)

n_before = df_no_same_station.height

df_no_same_station = df_no_same_station.filter(
    ~pl.col("Start station number").is_in(stations_to_remove)
    & ~pl.col("End station number").is_in(stations_to_remove)
)

print(f"Trips removed: {n_before - df_no_same_station.height}")
print(f"Remaining trips: {df_no_same_station.height:,}")
print(f"Distinct start stations: {df_no_same_station['Start station number'].n_unique()}")
print(f"Distinct end stations:   {df_no_same_station['End station number'].n_unique()}")

> **[CLAUDE - ADDED - recommended, optional]** Very long trips.
>
> The brief only asks to remove null/negative durations, but the duration column also has an extreme upper tail: **~5,300 trips last more than 24 hours** (the longest one lasts ~209 days). These are not real journeys (bike not docked properly, lost/stolen bike, maintenance) and they heavily distort the *mean* trip duration per station (e.g. station 300021 had a mean duration of ~150 min while its median is ~19 min). In notebook 2, `mean_duration` and `std_duration` were driving PC2 mostly because of these few trips.
>
> We remove trips longer than 24h (≈0.06% of the data). If you prefer to keep them, simply skip/delete the next cell — everything else still runs.

In [ ]:
# [CLAUDE - ADDED - optional] ------------------------------------------------
# Remove trips longer than 24 hours
# -----------------------------------------------------------------------------
MAX_DURATION_MS = 24 * 60 * 60 * 1000  # 24h in milliseconds

long_trips = df_no_same_station.filter(pl.col("Total duration (ms)") > MAX_DURATION_MS)
print(f"Trips longer than 24h: {long_trips.height:,} ({long_trips.height / df_no_same_station.height:.3%} of the trips)")
print("Duration of these trips (hours):")
print((long_trips["Total duration (ms)"] / 3_600_000).describe())

df_no_same_station = df_no_same_station.filter(pl.col("Total duration (ms)") <= MAX_DURATION_MS)
print(f"Remaining trips: {df_no_same_station.height:,}")

## Raw data transformation
In this section we will do a first data transformation that will be useful for the coming points. We will first make sure to load and maintain the data in the correct format, specially for the dates. We will then add date and time related features such as date of the trip, day of week (refering to the trip being made on monday, tuesday, etc) and hour of day.

In [18]:
#######################
# Add date columns
#######################
#Transform start and end date from text to date time
df = df_no_same_station.with_columns(
    pl.col("Start date").str.to_datetime("%Y-%m-%d %H:%M"),
    pl.col("End date").str.to_datetime("%Y-%m-%d %H:%M")
)

#Add Operating date, day of week and hour of day columns
df = df.with_columns(
    pl.col("Start date").dt.date().alias("Operating date"),
    pl.col("Start date").dt.weekday().alias("Day of week"),
    pl.col("Start date").dt.hour().alias("Hour of day"),
)

print(df)

shape: (8_762_355, 8)
┌────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────┬───────────┐
│ Start date ┆ Start      ┆ End date   ┆ End        ┆ Total      ┆ Operating  ┆ Day of ┆ Hour of   │
│ ---        ┆ station    ┆ ---        ┆ station    ┆ duration   ┆ date       ┆ week   ┆ day       │
│ datetime[μ ┆ number     ┆ datetime[μ ┆ number     ┆ (ms)       ┆ ---        ┆ ---    ┆ ---       │
│ s]         ┆ ---        ┆ s]         ┆ ---        ┆ ---        ┆ date       ┆ i8     ┆ i8        │
│            ┆ i64        ┆            ┆ i64        ┆ i64        ┆            ┆        ┆           │
╞════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════╪═══════════╡
│ 2025-01-14 ┆ 1043       ┆ 2025-01-15 ┆ 200149     ┆ 870468     ┆ 2025-01-14 ┆ 2      ┆ 23        │
│ 23:59:00   ┆            ┆ 00:13:00   ┆            ┆            ┆            ┆        ┆           │
│ 2025-01-14 ┆ 300015     ┆ 2025-01-15 ┆ 300229     ┆ 307181     ┆ 20

We will also add the stations cordinates in longitude and latitude format so that we can later use it to locate stations in interactive maps.

In [ ]:
#######################
# Add station coordinates
#######################
cord = pl.read_csv("../data/tfl_bike_stations.csv")

print(cord)

#Join start station
# [CLAUDE - CHANGED] removed suffix='depart_' : there is no column name clash on this join, so the suffix was never used
df = df.join(cord, left_on='Start station number', right_on = 'station_number', how='left')
df = df.rename({"latitude": "start_station_latitude",
               "longitude": "start_station_longitude",
               "station_name": "start_station_name"})

#Join end station
df = df.join(cord, left_on='End station number', right_on = 'station_number', how='left')
df = df.rename({"latitude": "end_station_latitude",
               "longitude": "end_station_longitude",
               "station_name": "end_station_name"})

df

> **[CLAUDE - ADDED]** Not every station id of the trip data exists in `tfl_bike_stations.csv`. The cell below lists the stations without coordinates: they stay in the analysis (their trips are valid) but they **cannot be drawn on the maps** of notebook 2. Many of them have ids ending in `444` (e.g. 200083444, 11274444), which look like temporary / relocated versions of existing stations — worth mentioning in the report as an identifier inconsistency (point 2.2 of the brief).

In [ ]:
# [CLAUDE - ADDED] stations without coordinates / name
all_station_ids = pl.concat([df["Start station number"], df["End station number"]]).unique()
missing_coords = all_station_ids.filter(~all_station_ids.is_in(cord["station_number"].implode())).sort()

trips_without_coords = df.filter(
    pl.col("start_station_latitude").is_null() | pl.col("end_station_latitude").is_null()
).height

print(f"Stations in the trip data: {all_station_ids.len()}")
print(f"Stations without coordinates: {missing_coords.len()}")
print(missing_coords.to_list())
print(f"Trips touching one of these stations: {trips_without_coords:,} ({trips_without_coords / df.height:.2%})")

In [20]:
#######################
#export df
#######################

#df.write_csv("../data/output/v2/2.clean_df_v2.csv")
df.write_parquet("../data/output/v2/2.clean_df_v2.parquet")
